# 🏺 Egyptian Hieroglyph Recognition Pipeline — v3
### Fuentes-Ferrer et al. (2025) — Applied Soft Computing
#### **Notebook 1/2** — Data Pre-Processing + Segmentation (IGSM)

**Paper:** https://doi.org/10.1016/j.asoc.2025.112793  
**Dataset:** https://github.com/rfuentesfe/EgyptianHieroglyphicText

---


---
# 📦 Step 1 — Data Pre-Processing
> تحميل Franken + Paper datasets، دمجهم في Glyph2025، resize آمن بدون overwrite، وفلترة الـ 164 class.
---


### Cell 1.1 — Imports & Global Config

In [ ]:
import os, random, json, time, warnings, shutil, subprocess, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import torchvision.transforms as T
from PIL import Image
from collections import Counter
from dataclasses import dataclass, field   # ✅ Fix: @dataclass
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
set_seed()

# ── Paths ────────────────────────────────────────────────────
WORK_DIR       = "/kaggle/working"
GLYPH2025_RAW  = os.path.join(WORK_DIR, "Glyph2025_raw")        # raw merged
GLYPH2025_DIR  = os.path.join(WORK_DIR, "Glyph2025_processed")  # ✅ Fix: separate processed dir
FRANKEN_DIR    = os.path.join(WORK_DIR, "GlyphFranken2025")
HIERO_DIR      = os.path.join(WORK_DIR, "GlyphHiero2025")

# ── Image settings ───────────────────────────────────────────
IMG_SIZE = 256
IMG_EXTS = (".jpg",".jpeg",".png",".bmp",".webp")
MEAN     = [0.485, 0.456, 0.406]
STD      = [0.229, 0.224, 0.225]
device   = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅  Device : {device}")
print(f"    WORK_DIR : {WORK_DIR}")


### Cell 1.2 — Download Franken Dataset

In [ ]:
FRANKEN_URL = "http://jvgemert.github.io/pub/EgyptianHieroglyphDataset.tar.gz"
TAR_PATH    = os.path.join(WORK_DIR, "EgyptianHieroglyphDataset.tar.gz")

def _find_class_root(base_dir: str) -> str:
    """
    الـ Franken tar ممكن يكون فيه أي structure:
      • base/A1/ A2/ ...          (flat — classes directly)
      • base/train/A1/ + base/test/A1/
      • base/images/A1/ ...
    الدالة دي بتدور على أول directory فيها class folders (أسماء هيروغليف).
    """
    import re
    pattern = re.compile(r'^[A-Za-z]{1,3}\d+$')
    for root, dirs, _ in os.walk(base_dir):
        glyph_dirs = [d for d in dirs if pattern.match(d)]
        if len(glyph_dirs) >= 5:
            return root
    return base_dir

def _collect_images_to(src_dir: str, dst_dir: str):
    """نقل كل الصور من src_dir لـ dst_dir مع lowercase."""
    os.makedirs(dst_dir, exist_ok=True)
    for cls in sorted(os.listdir(src_dir)):
        cls_path = os.path.join(src_dir, cls)
        if not os.path.isdir(cls_path): continue
        cls_dst = os.path.join(dst_dir, cls.lower())
        os.makedirs(cls_dst, exist_ok=True)
        for f in os.listdir(cls_path):
            if f.lower().endswith(IMG_EXTS):
                shutil.copy2(os.path.join(cls_path, f),
                             os.path.join(cls_dst, f.lower()))

if not os.path.exists(FRANKEN_DIR):
    print("⏳ Downloading Franken dataset ...")
    urllib.request.urlretrieve(FRANKEN_URL, TAR_PATH)
    print("✅  Downloaded. Extracting ...")

    EXTRACT_DIR = os.path.join(WORK_DIR, "EgyptianHieroglyphDataset_raw")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with tarfile.open(TAR_PATH) as t:
        t.extractall(EXTRACT_DIR)

    # ── دور على الـ structure الحقيقية ──────────────────────
    print("   Detecting dataset structure ...")
    contents = os.listdir(EXTRACT_DIR)
    print(f"   Top-level contents: {contents[:10]}")

    # لو subfolder واحد بس → ادخل جوّاه
    if len(contents) == 1 and os.path.isdir(os.path.join(EXTRACT_DIR, contents[0])):
        inner = os.path.join(EXTRACT_DIR, contents[0])
    else:
        inner = EXTRACT_DIR

    class_root = _find_class_root(inner)
    print(f"   Class root detected: {class_root}")

    # ── دمج train/ + test/ لو موجودين ─────────────────────
    train_dir = os.path.join(class_root, "train")
    test_dir  = os.path.join(class_root, "test")
    os.makedirs(FRANKEN_DIR, exist_ok=True)

    if os.path.exists(train_dir):
        print("   Found train/test structure → merging ...")
        _collect_images_to(train_dir, FRANKEN_DIR)
        if os.path.exists(test_dir):
            _collect_images_to(test_dir, FRANKEN_DIR)
    else:
        print("   Found flat structure → copying classes directly ...")
        _collect_images_to(class_root, FRANKEN_DIR)

    # Cleanup
    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)
    if os.path.exists(TAR_PATH): os.remove(TAR_PATH)

    n_cls = len([d for d in os.listdir(FRANKEN_DIR)
                 if os.path.isdir(os.path.join(FRANKEN_DIR,d))])
    n_img = sum(len(os.listdir(os.path.join(FRANKEN_DIR,c)))
                for c in os.listdir(FRANKEN_DIR)
                if os.path.isdir(os.path.join(FRANKEN_DIR,c)))
    print(f"✅  Franken ready → {FRANKEN_DIR}")
    print(f"   Classes: {n_cls} | Images: {n_img:,}")
else:
    print(f"✅  Franken already exists → {FRANKEN_DIR}")


### Cell 1.3 — Download Paper Dataset (GitHub)

In [ ]:
GITHUB_REPO = "https://github.com/rfuentesfe/EgyptianHieroglyphicText.git"
REPO_DIR    = os.path.join(WORK_DIR, "EgyptianHieroglyphicText")

if not os.path.exists(HIERO_DIR):
    print("⏳ Cloning paper dataset from GitHub ...")
    # ✅ Fix: consistent subprocess usage (no os.system anywhere)
    subprocess.run(["git","clone","--depth=1", GITHUB_REPO, REPO_DIR],
                   check=True, capture_output=True)
    raw_ds = os.path.join(REPO_DIR, "dataset")
    if not os.path.exists(raw_ds):
        raw_ds = REPO_DIR
    shutil.copytree(raw_ds, HIERO_DIR)
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    print(f"✅  Paper dataset ready → {HIERO_DIR}")
else:
    print(f"✅  Paper dataset already exists → {HIERO_DIR}")


### Cell 1.4 — Merge Datasets → Glyph2025_raw

In [ ]:
def merge_datasets(sources, dst):
    os.makedirs(dst, exist_ok=True)
    total = 0
    for src in sources:
        if not os.path.exists(src):
            print(f"  ⚠️  Source not found: {src} — skipping"); continue
        for cls in os.listdir(src):
            cls_src = os.path.join(src, cls)
            if not os.path.isdir(cls_src): continue
            cls_dst = os.path.join(dst, cls.lower())
            os.makedirs(cls_dst, exist_ok=True)
            for f in os.listdir(cls_src):
                if f.lower().endswith(IMG_EXTS):
                    shutil.copy2(os.path.join(cls_src,f),
                                 os.path.join(cls_dst, f.lower()))
                    total += 1
    return total

if not os.path.exists(GLYPH2025_RAW):
    print("⏳ Merging datasets ...")
    n = merge_datasets([FRANKEN_DIR, HIERO_DIR], GLYPH2025_RAW)
    print(f"✅  Glyph2025_raw created — {n:,} images")
else:
    print(f"✅  Glyph2025_raw already exists → {GLYPH2025_RAW}")


### Cell 1.5 — Safe Resize & Pad (✅ Fix: separate output dir)

In [ ]:
def resize_and_pad(src_path, dst_path, size=224):
    """
    ✅ Fix: saves to dst_path (different from src_path).
    No in-place overwrite → idempotent, can re-run safely.
    """
    try:
        img = Image.open(src_path).convert("RGB")
        img.thumbnail((size, size), Image.LANCZOS)
        new_img = Image.new("RGB", (size, size), (0, 0, 0))
        new_img.paste(img, ((size-img.width)//2, (size-img.height)//2))
        os.makedirs(os.path.dirname(dst_path), exist_ok=True)
        new_img.save(dst_path)
    except Exception as e:
        print(f"  ⚠️  Skip {src_path}: {e}")

if not os.path.exists(GLYPH2025_DIR):
    print("⏳ Resizing all images → Glyph2025_processed/ ...")
    pairs = []
    for cls in os.listdir(GLYPH2025_RAW):
        cls_src = os.path.join(GLYPH2025_RAW, cls)
        cls_dst = os.path.join(GLYPH2025_DIR, cls)
        if not os.path.isdir(cls_src): continue
        for f in os.listdir(cls_src):
            if f.lower().endswith(IMG_EXTS):
                pairs.append((os.path.join(cls_src,f), os.path.join(cls_dst,f)))

    for src,dst in tqdm(pairs, desc="Resizing"):
        resize_and_pad(src, dst, size=224)
    print(f"✅  {len(pairs):,} images resized → {GLYPH2025_DIR}")
else:
    n = sum(len(os.listdir(os.path.join(GLYPH2025_DIR,c)))
            for c in os.listdir(GLYPH2025_DIR)
            if os.path.isdir(os.path.join(GLYPH2025_DIR,c)))
    print(f"✅  Glyph2025_processed already exists ({n:,} images)")


### Cell 1.6 — Class Filter & Build DataFrame

In [ ]:
OFFICIAL_CLASSES = {
    "a1","a2","a19","a24","a30","a40","a42","a50","b1",
    "d1","d2","d4","d21","d28","d35","d36","d37","d39","d40",
    "d45","d46","d52","d54","d55","d56","d58","d60",
    "e1","e7","e21","e23","e34",
    "f1","f4","f12","f13","f18","f21","f26","f31","f32","f34","f35","f39",
    "g1","g5","g7","g14","g17","g25","g35","g36","g37","g38","g39","g40","g43",
    "h1","h6","h8","i1","i9","i10","l1","l2",
    "m2","m3","m12","m16","m17","m18","m20","m22","m23","m42",
    "n1","n5","n8","n14","n18","n25","n26","n29","n30","n31",
    "n33","n35","n36","n37","n42",
    "o1","o3","o4","o6","o28","o29","o34","o49","o50","p8",
    "q1","q2","q3","r4","r7","r8","r11","r14",
    "s3","s19","s21","s24","s27","s28","s29","s34","s38","s40","s42",
    "t21","t22","t28","u1","u6","u7","u15","u23","u33",
    "v1","v4","v6","v7","v10","v13","v20","v28","v29","v30","v31",
    "w11","w14","w15","w17","w18","w19","w22","w23","w24","w25",
    "x1","x7","x8","y1","y2","y3","y4","y5",
    "z1","z2","z3","z4","z11","aa13","aa15",
}
MIN_SAMPLES = 7

print("── Scanning Glyph2025_processed ──")
class_counts = {}
for cls in sorted(os.listdir(GLYPH2025_DIR)):
    p = os.path.join(GLYPH2025_DIR, cls)
    if not os.path.isdir(p): continue
    class_counts[cls] = sum(1 for f in os.listdir(p) if f.lower().endswith(IMG_EXTS))

kept          = {c:n for c,n in class_counts.items() if n >= MIN_SAMPLES}
final_classes = sorted([c for c in kept if c in OFFICIAL_CLASSES])
print(f"Total classes : {len(class_counts)} | After min-{MIN_SAMPLES} : {len(kept)} | Paper filter : {len(final_classes)}")

paths, labels = [], []
for cls in final_classes:
    for f in os.listdir(os.path.join(GLYPH2025_DIR, cls)):
        if f.lower().endswith(IMG_EXTS):
            paths.append(os.path.join(GLYPH2025_DIR,cls,f)); labels.append(cls)

df           = pd.DataFrame({"path":paths,"label":labels})
classes      = sorted(df["label"].unique().tolist())
class_to_idx = {c:i for i,c in enumerate(classes)}
idx_to_class = {i:c for c,i in class_to_idx.items()}
df["y"]      = df["label"].map(class_to_idx)
print(f"\n✅  Final: {len(classes)} classes | {len(df):,} images")


### Cell 1.7 — 📊 Dataset Stats Dashboard (New Visualization)

In [ ]:
cnt = df["label"].value_counts().sort_index()
vals = cnt.values

fig = plt.figure(figsize=(22, 14))
fig.suptitle("📊 Glyph2025 — Dataset Statistics Dashboard", fontsize=16, fontweight="bold", y=0.98)

# ── Plot 1: Class Distribution Bar ───────────────────────────
ax1 = fig.add_subplot(3,2,(1,2))
colors_bar = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(cnt)))
bars = ax1.bar(cnt.index, cnt.values, color=colors_bar, edgecolor="none")
ax1.axhline(vals.mean(), color="dodgerblue", lw=1.5, ls="--", label=f"Mean = {vals.mean():.0f}")
ax1.axhline(MIN_SAMPLES,  color="red",        lw=1,   ls=":",  label=f"Min threshold = {MIN_SAMPLES}")
ax1.set_title(f"Class Distribution — {len(cnt)} classes, {len(df):,} total images", fontsize=12)
ax1.set_xlabel("Gardiner Code"); ax1.set_ylabel("# Images")
ax1.legend(fontsize=9); plt.sca(ax1); plt.xticks(rotation=90, fontsize=6)

# ── Plot 2: Images per Class Histogram ───────────────────────
ax2 = fig.add_subplot(3,2,3)
ax2.hist(vals, bins=30, color="steelblue", edgecolor="white", alpha=0.85)
ax2.axvline(vals.mean(),   color="dodgerblue", lw=2, ls="--", label=f"Mean={vals.mean():.0f}")
ax2.axvline(np.median(vals), color="tomato",   lw=2, ls="--", label=f"Median={np.median(vals):.0f}")
ax2.set_title("Distribution of Class Sizes"); ax2.set_xlabel("# Images per Class"); ax2.set_ylabel("# Classes")
ax2.legend(fontsize=9)

# ── Plot 3: Top 15 & Bottom 15 classes ───────────────────────
ax3 = fig.add_subplot(3,2,4)
sorted_cnt = cnt.sort_values(ascending=False)
top15    = sorted_cnt.head(15)
bottom15 = sorted_cnt.tail(15)
ax3.barh(list(bottom15.index) + ["..."] + list(top15.index),
         list(bottom15.values) + [0] + list(top15.values),
         color=["tomato"]*15 + ["gray"] + ["steelblue"]*15)
ax3.set_title("Top 15 (blue) vs Bottom 15 (red) Classes")
ax3.set_xlabel("# Images"); ax3.invert_yaxis()
top_patch = mpatches.Patch(color="steelblue", label="Top 15")
bot_patch = mpatches.Patch(color="tomato",    label="Bottom 15")
ax3.legend(handles=[top_patch, bot_patch], fontsize=9)

# ── Plot 4: Category letter breakdown ────────────────────────
ax4 = fig.add_subplot(3,2,5)
letter_counts = {}
for cls in final_classes:
    letter = cls[0] if cls[0].isalpha() else cls[:2]
    letter_counts[letter] = letter_counts.get(letter, 0) + class_counts.get(cls, 0)
lk = sorted(letter_counts.keys())
lv = [letter_counts[k] for k in lk]
ax4.bar(lk, lv, color=plt.cm.tab20(np.linspace(0,1,len(lk))), edgecolor="none")
ax4.set_title("Images per Gardiner Category (Letter)")
ax4.set_xlabel("Category Letter"); ax4.set_ylabel("# Images")
plt.sca(ax4); plt.xticks(rotation=45, fontsize=9)

# ── Plot 5: Cumulative coverage ───────────────────────────────
ax5 = fig.add_subplot(3,2,6)
sorted_vals = np.sort(vals)[::-1]
cumsum = np.cumsum(sorted_vals) / sorted_vals.sum() * 100
ax5.plot(range(1, len(cumsum)+1), cumsum, color="mediumseagreen", lw=2)
ax5.axhline(80, color="orange", ls="--", lw=1.2, label="80% coverage")
ax5.axhline(95, color="red",    ls="--", lw=1.2, label="95% coverage")
idx80 = np.searchsorted(cumsum, 80)
idx95 = np.searchsorted(cumsum, 95)
ax5.axvline(idx80, color="orange", ls=":", lw=1, label=f"80% at class {idx80}")
ax5.axvline(idx95, color="red",    ls=":", lw=1, label=f"95% at class {idx95}")
ax5.set_title("Cumulative Image Coverage by Class Rank")
ax5.set_xlabel("# Top Classes"); ax5.set_ylabel("Cumulative %")
ax5.legend(fontsize=8); ax5.set_ylim(0,105)

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,"dataset_dashboard.png"), dpi=120, bbox_inches="tight")
plt.show()
print(f"\n📊 Summary:")
print(f"   Classes : {len(classes)} | Images : {len(df):,}")
print(f"   Min/Max : {vals.min()} / {vals.max()} | Mean : {vals.mean():.1f} | Std : {vals.std():.1f}")
print(f"   Imbalance ratio : {vals.max()/vals.min():.1f}x  ← class weights needed!")
print("\n✅  Step 1 complete — Dataset ready!")


---
# 🔍 Step 2 — Segmentation (IGSM + MBRS) — Smart Edition ✅
> استخراج الهيروغليفية من صور اللوحات الحجرية.
>
> **Pipeline:** Border Removal → Multi-threshold Binary (Otsu + Adaptive) → Stroke Detection → Watershed → SAM → NMS → Reading Order
>
> **v3 Improvements:**
> - ✅ `SamAutomaticMaskGenerator` بدل Point-based (بيغطي كل الصورة)
> - ✅ Adaptive Threshold بدل Otsu وحده
> - ✅ Solidity + Aspect Ratio filter للـ masks
> - ✅ Row Clustering للترتيب الصحيح RTL
> - ✅ تحليل شامل لكل الصور في الداتاسيت
---


### Cell 2.1 — Segmentation Imports

In [ ]:
import cv2
import glob as _glob
from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from segment_anything import sam_model_registry, SamPredictor, SamAutomaticMaskGenerator
from collections import defaultdict
print("✅  Segmentation imports OK")


### Cell 2.2 — SegConfig

In [ ]:
@dataclass
class SegConfig:
    # SAM
    sam_checkpoint             : str   = "/kaggle/working/sam_vit_b.pth"
    seg_output_dir             : str   = "/kaggle/working/segmented_glyphs"

    # Border removal
    border_thresh              : int   = 35

    # Denoising
    denoise_h                  : int   = 10
    denoise_template           : int   = 7
    denoise_search             : int   = 21

    # CLAHE
    clahe_clip                 : float = 2.5          # ✅ أقل حدة للصور الحجرية
    clahe_grid                 : tuple = field(default_factory=lambda: (8, 8))

    # Threshold
    adaptive_block             : int   = 25
    adaptive_c                 : int   = 8            # ✅ أعلى قليلاً لتقليل الضوضاء

    # Noise & area
    min_area                   : int   = 80

    # Stroke detector Z1/Z2/Z3
    stroke_min_aspect          : float = 3.5
    stroke_min_width           : int   = 15
    stroke_max_height          : int   = 40
    stroke_min_area            : int   = 60
    stroke_min_solidity        : float = 0.35
    stroke_group_gap           : int   = 90
    parallel_x_overlap         : float = 0.60
    stroke_count_check         : bool  = True
    max_stroke_diff            : int   = 0
    parallel_separator_min_area: int   = 200

    # Size filter
    min_mask_width_ratio       : float = 0.03         # ✅ أقل من القديم لالتقاط رموز صغيرة
    min_mask_height_ratio      : float = 0.04
    noise_min_density          : float = 0.18
    max_area_ratio             : float = 0.40

    # Watershed
    watershed_dist             : int   = 5

    # SAM Auto Generator ✅ جديد
    points_per_side            : int   = 32
    pred_iou_thresh            : float = 0.82
    stability_score_thresh     : float = 0.90
    min_mask_region_area       : int   = 80
    box_nms_thresh             : float = 0.50         # ✅ أعلى من 0.3 القديم

    # Mask quality filters ✅ جديد
    min_solidity               : float = 0.30
    max_aspect_ratio           : float = 8.0

    # NMS
    iou_threshold              : float = 0.35

    # Crops
    padding                    : int   = 6
    bg_fill                    : str   = "white"
    output_size                : tuple = field(default_factory=lambda: (128, 128))

    # Reading order clustering ✅ جديد
    row_tolerance_ratio        : float = 0.05

seg_cfg = SegConfig()
print("✅  SegConfig ready")
print(seg_cfg)


### Cell 2.3 — Load SAM (Automatic Mode) ✅

In [ ]:
def load_sam_auto(cfg: SegConfig) -> SamAutomaticMaskGenerator:
    """
    ✅ SamAutomaticMaskGenerator بدل SamPredictor:
    بيمسح الصورة كلها بـ grid منتظم → بيكتشف رموز بيفوّتها الـ point-based approach.
    """
    if not os.path.exists(cfg.sam_checkpoint):
        print("⏳ Downloading SAM ViT-B ...")
        subprocess.run([
            "wget", "-q",
            "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
            "-O", cfg.sam_checkpoint
        ], check=True)
    device_sam = "cuda" if torch.cuda.is_available() else "cpu"
    sam = sam_model_registry["vit_b"](checkpoint=cfg.sam_checkpoint)
    sam.to(device=device_sam)

    generator = SamAutomaticMaskGenerator(
        model                  = sam,
        points_per_side        = cfg.points_per_side,
        pred_iou_thresh        = cfg.pred_iou_thresh,
        stability_score_thresh = cfg.stability_score_thresh,
        min_mask_region_area   = cfg.min_mask_region_area,
        box_nms_thresh         = cfg.box_nms_thresh,
    )
    print(f"✅  SAM Auto Generator loaded on [{device_sam}]")
    return generator

sam_generator = load_sam_auto(seg_cfg)

# ✅ SamPredictor أيضاً للـ fallback في run_single
def load_sam_predictor(cfg: SegConfig) -> SamPredictor:
    device_sam = "cuda" if torch.cuda.is_available() else "cpu"
    sam = sam_model_registry["vit_b"](checkpoint=cfg.sam_checkpoint)
    sam.to(device=device_sam)
    return SamPredictor(sam)

sam_predictor = load_sam_predictor(seg_cfg)
print("✅  Both generator + predictor ready")


### Cell 2.4 — Border Removal & Color Detection

In [ ]:
def load_image(path: str) -> np.ndarray:
    img = cv2.imread(path)
    if img is None: raise FileNotFoundError(f"Not found: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _detect_glyph_color(image: np.ndarray) -> str:
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    dark_mask = hsv[:,:,2] < 100
    if dark_mask.sum() == 0: return "dark"
    return "color" if hsv[:,:,1][dark_mask].mean() > 40 else "dark"

def remove_border(image: np.ndarray, cfg: SegConfig) -> np.ndarray:
    """Remove colored museum-frame borders using HSV saturation."""
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV); sat = hsv[:,:,1]
    H,W = image.shape[:2]; margin = max(8, min(H,W)//20)
    edges = {"top":sat[:margin,:].mean(), "bottom":sat[-margin:,:].mean(),
             "left":sat[:,:margin].mean(), "right":sat[:,-margin:].mean()}
    high  = {k:v for k,v in edges.items() if v > cfg.border_thresh}
    if not high: return image
    y1 = margin if "top"    in high else 0
    y2 = H-margin if "bottom" in high else H
    x1 = margin if "left"   in high else 0
    x2 = W-margin if "right"  in high else W
    c = image[y1:y2, x1:x2]
    return c if c.shape[0]>=30 and c.shape[1]>=30 else image

print("✅  Border removal ready")


### Cell 2.5 — Multi-Threshold Preprocessing ✅

In [ ]:
def preprocess(image: np.ndarray, cfg: SegConfig) -> tuple:
    """
    ✅ Multi-threshold fusion: Otsu + Adaptive Gaussian + Adaptive Mean
    OR of all 3 → يلتقط رموز بيفوّتها أي threshold منفرد.
    """
    gray     = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    clean    = cv2.fastNlMeansDenoising(gray, None, cfg.denoise_h,
                                         cfg.denoise_template, cfg.denoise_search)
    clahe    = cv2.createCLAHE(clipLimit=cfg.clahe_clip, tileGridSize=cfg.clahe_grid)
    enhanced = clahe.apply(clean)
    _, otsu  = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)

    if _detect_glyph_color(image) == "color":
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        combined = cv2.bitwise_or(
            cv2.bitwise_and((hsv[:,:,1]>50).astype(np.uint8)*255,
                            (hsv[:,:,2]<160).astype(np.uint8)*255), otsu)
    else:
        ag = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                    cv2.THRESH_BINARY_INV, cfg.adaptive_block, cfg.adaptive_c)
        am = cv2.adaptiveThreshold(enhanced, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                                    cv2.THRESH_BINARY_INV, cfg.adaptive_block+10, cfg.adaptive_c-2)
        combined = cv2.bitwise_or(cv2.bitwise_or(otsu, ag), am)

    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    return cv2.morphologyEx(cv2.morphologyEx(combined, cv2.MORPH_CLOSE, k, iterations=1),
                             cv2.MORPH_OPEN, k, iterations=1), enhanced

print("✅  Preprocessing ready")


### Cell 2.6 — Noise Removal & Stroke Detection (Z1/Z2/Z3)

In [ ]:
def remove_noise(binary: np.ndarray, cfg: SegConfig) -> np.ndarray:
    n,labels,stats,_ = cv2.connectedComponentsWithStats(binary)
    out = np.zeros_like(binary)
    for i in range(1, n):
        w=stats[i,cv2.CC_STAT_WIDTH]; h=stats[i,cv2.CC_STAT_HEIGHT]; a=stats[i,cv2.CC_STAT_AREA]
        if w>10 and h>10 and a>cfg.min_area and a/max(w*h,1)>=cfg.noise_min_density:
            out[labels==i] = 255
    return out


def detect_strokes(binary: np.ndarray, cfg: SegConfig) -> tuple:
    """Detects horizontal strokes (Z1/Z2/Z3). stroke_count_check يمنع دمج Z2 مع Z3."""
    n,labels,stats,_ = cv2.connectedComponentsWithStats(binary)
    H,W = binary.shape
    raw, sid = [], set()

    for i in range(1, n):
        x=stats[i,cv2.CC_STAT_LEFT]; y=stats[i,cv2.CC_STAT_TOP]
        w=stats[i,cv2.CC_STAT_WIDTH]; h=stats[i,cv2.CC_STAT_HEIGHT]; a=stats[i,cv2.CC_STAT_AREA]
        if not (w and h and a>=cfg.stroke_min_area): continue
        roi = (labels[y:y+h, x:x+w]==i).astype(np.uint8)*255
        cnts,_ = cv2.findContours(roi, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        sol = 0.0
        if cnts:
            ha = cv2.contourArea(cv2.convexHull(cnts[0]))
            if ha > 0: sol = a/ha
        if (w/h >= cfg.stroke_min_aspect and w >= cfg.stroke_min_width
                and h <= cfg.stroke_max_height and sol >= cfg.stroke_min_solidity):
            m = np.zeros(binary.shape, dtype=np.uint8); m[labels==i] = 1
            raw.append({"mask":m,"x1":x,"y1":y,"x2":x+w,"y2":y+h}); sid.add(i)

    if not raw: return [], binary.copy()
    raw.sort(key=lambda s: s["y1"])

    def xov(a, b):
        ov = max(0, min(a["x2"],b["x2"]) - max(a["x1"],b["x1"]))
        return ov / max(min(a["x2"]-a["x1"], b["x2"]-b["x1"]), 1)

    p = list(range(len(raw)))
    def find(i):
        while p[i]!=i: p[i]=p[p[i]]; i=p[i]
        return i
    def union(i,j):
        pi,pj=find(i),find(j)
        if pi!=pj: p[pi]=pj

    for i in range(len(raw)):
        for j in range(i+1, len(raw)):
            if xov(raw[i], raw[j]) >= cfg.parallel_x_overlap: union(i,j)

    pg = defaultdict(list)
    for i,s in enumerate(raw): pg[find(i)].append(s)
    fg = []
    for grp in pg.values():
        grp.sort(key=lambda s:s["y1"]); cur=[grp[0]]
        for s in grp[1:]:
            if s["y1"]-cur[-1]["y2"] <= max(cfg.stroke_group_gap, int(H*0.12)): cur.append(s)
            else: fg.append(cur); cur=[s]
        fg.append(cur)

    def gb(g): return min(s["x1"] for s in g),min(s["y1"] for s in g),max(s["x2"] for s in g),max(s["y2"] for s in g)
    def gxov(a,b):
        ax1,_,ax2,_ = gb(a); bx1,_,bx2,_ = gb(b)
        ov = max(0, min(ax2,bx2)-max(ax1,bx1))
        return ov / max(min(ax2-ax1, bx2-bx1), 1)

    fg.sort(key=lambda g: gb(g)[1])
    fp = list(range(len(fg)))
    def ff(i):
        while fp[i]!=i: fp[i]=fp[fp[i]]; i=fp[i]
        return i
    def fu(i,j):
        pi,pj=ff(i),ff(j)
        if pi!=pj: fp[pi]=pj

    for i in range(len(fg)):
        for j in range(i+1, len(fg)):
            if cfg.stroke_count_check and abs(len(fg[i])-len(fg[j])) > cfg.max_stroke_diff: continue
            if gxov(fg[i], fg[j]) < cfg.parallel_x_overlap: continue
            if gb(fg[i])[3] >= gb(fg[j])[1]: continue
            fu(i,j)

    mfg = defaultdict(list)
    for i,g in enumerate(fg): mfg[ff(i)].extend(g)
    stroke_masks = []
    for grp in mfg.values():
        c = np.zeros(binary.shape, dtype=np.uint8)
        for s in grp: c = np.logical_or(c, s["mask"]).astype(np.uint8)
        stroke_masks.append(c)

    rem = binary.copy()
    for i in sid: rem[labels==i] = 0
    return stroke_masks, rem

print("✅  Stroke detection ready")


### Cell 2.7 — SAM Auto Segmentation + Solidity Filter ✅

In [ ]:
def _compute_solidity(mask: np.ndarray) -> float:
    """Solidity = area / convex_hull_area — يكشف الـ masks الغريبة الشكل."""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    hull_area = cv2.contourArea(cv2.convexHull(cnt))
    return float(area / hull_area) if hull_area > 0 else 0.0


def sam_auto_segment(image: np.ndarray, generator: SamAutomaticMaskGenerator,
                     cfg: SegConfig) -> list:
    """
    ✅ SamAutomaticMaskGenerator:
    - بيمسح الصورة كلها بـ grid 32×32 نقطة
    - بيرجع masks بـ predicted_iou score لكل واحدة
    - بعدين بنفلتر بـ Solidity + Aspect Ratio
    """
    H, W   = image.shape[:2]
    img_area = H * W
    min_area = img_area * 0.0002
    max_area = img_area * cfg.max_area_ratio

    try:
        raw_masks = generator.generate(image)
    except Exception as e:
        print(f"  ⚠️  SAM Auto failed: {e}")
        return []

    filtered = []
    for m_data in raw_masks:
        mask = m_data["segmentation"].astype(np.uint8)
        area = float(mask.sum())
        if not (min_area <= area <= max_area): continue

        ys, xs = np.where(mask)
        if len(xs) == 0: continue
        w = xs.max() - xs.min() + 1
        h = ys.max() - ys.min() + 1
        if (w / max(h, 1)) > cfg.max_aspect_ratio: continue     # يحذف الخطوط الطويلة

        sol = _compute_solidity(mask)
        if sol < cfg.min_solidity: continue                       # يحذف الـ masks الغريبة

        filtered.append(mask)

    return filtered


def watershed_separation(binary: np.ndarray, cfg: SegConfig) -> tuple:
    if binary.max() == 0: return binary, 0, np.zeros_like(binary, dtype=np.int32)
    dist   = ndi.distance_transform_edt(binary)
    coords = peak_local_max(dist, min_distance=cfg.watershed_dist, labels=binary)
    mp     = np.zeros(dist.shape, dtype=bool)
    if len(coords): mp[tuple(coords.T)] = True
    mk, _  = ndi.label(mp)
    ws     = watershed(-dist, mk, mask=binary)
    return (ws>0).astype(np.uint8)*255, int(ws.max()), ws

print("✅  SAM Auto Segment + Watershed ready")


### Cell 2.8 — Size Filter, NMS, Reading Order (RTL + Row Clustering) ✅ & Save Crops

In [ ]:
def size_filter(masks: list, shape: tuple, cfg: SegConfig) -> list:
    H,W = shape[:2]; max_a = H*W*cfg.max_area_ratio
    kept = []
    for m in masks:
        if not m.sum(): continue
        ys,xs = np.where(m==1)
        mw=int(xs.max()-xs.min()); mh=int(ys.max()-ys.min()); a=int(m.sum())
        if (a<max_a and mw>=W*cfg.min_mask_width_ratio and mh>=H*cfg.min_mask_height_ratio
                and a/max(mw*mh,1)>=cfg.noise_min_density):
            kept.append(m)
    return kept


def _iou(m1, m2) -> float:
    return float(np.logical_and(m1,m2).sum()) / (float(np.logical_or(m1,m2).sum())+1e-6)


def apply_nms(masks: list, cfg: SegConfig, expected_count: int=None) -> list:
    if not masks: return []
    def _run(thr):
        idx = sorted(range(len(masks)), key=lambda i: -int(masks[i].sum())); keep=[]
        for i in idx:
            if not any(_iou(masks[i], masks[j])>thr for j in keep): keep.append(i)
        return keep
    thr=cfg.iou_threshold; kept=_run(thr)
    if expected_count:
        for _ in range(20):
            if len(kept)<=expected_count: break
            thr+=0.05; kept=_run(thr)
    return [masks[i] for i in kept]


def _cluster_rows(data: list, H: int, tolerance_ratio: float) -> list:
    """✅ Row Clustering — يجمّع الرموز في صفوف بـ tolerance نسبي."""
    if not data: return []
    tol = H * tolerance_ratio
    data_s = sorted(data, key=lambda x: x["cy"])
    rows, cur, cy = [], [data_s[0]], data_s[0]["cy"]
    for item in data_s[1:]:
        if abs(item["cy"] - cy) <= tol: cur.append(item)
        else: rows.append(cur); cur = [item]; cy = item["cy"]
    rows.append(cur)
    return rows


def reading_order(image: np.ndarray, masks: list, cfg: SegConfig=None) -> list:
    """
    ✅ RTL Reading Order مع Row Clustering:
    - صفوف (W≥H): أعلى → أسفل، داخل كل صف يمين → يسار
    - أعمدة (H>W): يمين → يسار، داخل كل عمود أعلى → أسفل
    """
    if cfg is None: cfg = SegConfig()
    H, W = image.shape[:2]
    ann = []
    for m in masks:
        ys,xs = np.where(m==1)
        if not len(xs): continue
        x1,y1,x2,y2 = xs.min(),ys.min(),xs.max(),ys.max()
        ann.append(dict(mask=m, x1=int(x1),y1=int(y1),x2=int(x2),y2=int(y2),
                        cx=(x1+x2)/2, cy=(y1+y2)/2))
    if not ann: return []

    rows = _cluster_rows(ann, H, cfg.row_tolerance_ratio)
    ordered = []
    if H > W:   # أعمدة
        rows.sort(key=lambda r: -np.mean([g["cx"] for g in r]))
        for row in rows:
            row.sort(key=lambda g: g["cy"]); ordered.extend(row)
    else:       # صفوف
        rows.sort(key=lambda r: np.mean([g["cy"] for g in r]))
        for row in rows:
            row.sort(key=lambda g: -g["cx"]); ordered.extend(row)  # RTL ✅

    return [g["mask"] for g in ordered]


def save_crops(image: np.ndarray, masks: list, cfg: SegConfig, image_name="image") -> list:
    os.makedirs(cfg.seg_output_dir, exist_ok=True)
    H,W = image.shape[:2]; crops=[]
    base = os.path.splitext(os.path.basename(str(image_name)))[0]
    for idx,tm in enumerate(masks):
        ys,xs = np.where(tm==1)
        if not len(xs): continue
        x1=max(0,int(xs.min())-cfg.padding); y1=max(0,int(ys.min())-cfg.padding)
        x2=min(W,int(xs.max())+cfg.padding); y2=min(H,int(ys.max())+cfg.padding)
        er = np.zeros((H,W),dtype=np.uint8); er[y1:y2,x1:x2]=1; er[tm==1]=0
        crop = image[y1:y2,x1:x2].copy(); er_r=er[y1:y2,x1:x2].astype(bool)
        if cfg.bg_fill=="white": crop[er_r]=255
        oh,ow=crop.shape[:2]; side=max(oh,ow)
        sq=np.full((side,side,3),255,dtype=np.uint8)
        sq[(side-oh)//2:(side-oh)//2+oh,(side-ow)//2:(side-ow)//2+ow]=crop
        std=cv2.resize(sq, cfg.output_size, interpolation=cv2.INTER_AREA)
        cv2.imwrite(os.path.join(cfg.seg_output_dir, f"{base}_glyph_{idx+1:03d}.png"),
                    cv2.cvtColor(std, cv2.COLOR_RGB2BGR))
        crops.append(std)
    return crops

print("✅  NMS, Reading Order & Save Crops ready")


### Cell 2.9 — Visualization + run_single Pipeline

In [ ]:
def visualize_segmentation(image: np.ndarray, binary: np.ndarray,
                           masks: list, crops: list, title: str=""):
    """4-panel: Original | Binary | Glyphs detected | Crops grid"""
    numbered = image.copy()
    colors   = plt.cm.tab20.colors
    for idx,m in enumerate(masks):
        ys,xs = np.where(m==1)
        if not len(xs): continue
        x1,y1,x2,y2 = xs.min(),ys.min(),xs.max(),ys.max()
        c = tuple(int(v*255) for v in colors[idx%len(colors)][:3])
        cv2.rectangle(numbered,(x1,y1),(x2,y2),c,2)
        cv2.putText(numbered,str(idx+1),(x1+2,y1+14),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1)

    fig,axes = plt.subplots(1,3,figsize=(18,7))
    axes[0].imshow(image);              axes[0].set_title("① Original",fontsize=12);                       axes[0].axis("off")
    axes[1].imshow(binary,cmap="gray"); axes[1].set_title("② Binary Mask (Multi-threshold ✅)",fontsize=12); axes[1].axis("off")
    axes[2].imshow(numbered);           axes[2].set_title(f"③ {len(masks)} Glyphs Detected",fontsize=12);  axes[2].axis("off")
    plt.suptitle(f"IGSM Segmentation — {title}", fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.show()

    if crops:
        n=len(crops); cols=min(n,10); rows=(n+cols-1)//cols
        fig2,axes2 = plt.subplots(rows,cols,figsize=(cols*2, rows*2.2))
        axes2=np.array(axes2).flatten()
        for i,crop in enumerate(crops):
            axes2[i].imshow(crop); axes2[i].set_title(str(i+1),fontsize=9); axes2[i].axis("off")
        for ax in axes2[n:]: ax.axis("off")
        plt.suptitle(f"All Extracted Glyphs ({n} total)", fontsize=13, fontweight="bold")
        plt.tight_layout(); plt.show()


def run_single(image_path: str, cfg: SegConfig=None,
               generator: SamAutomaticMaskGenerator=None,
               predictor: SamPredictor=None,
               expected_glyphs: int=None, show: bool=True) -> tuple:
    """
    Full segmentation pipeline على صورة واحدة.
    ✅ يستخدم SAM Auto Generator أولاً، لو فشل يرجع للـ predictor.
    """
    if cfg is None: cfg = SegConfig()
    if generator is None and predictor is None:
        raise ValueError("Pass sam_generator or sam_predictor!")

    print(f"\n{'─'*50}\n📸  {os.path.basename(image_path)}")
    image  = load_image(image_path)
    image  = remove_border(image, cfg)
    print(f"  🎨 Type: {_detect_glyph_color(image)}")

    binary, _ = preprocess(image, cfg)
    binary    = remove_noise(binary, cfg)
    stroke_masks, no_strokes = detect_strokes(binary, cfg)
    print(f"  📏 Stroke groups: {len(stroke_masks)}")

    # ✅ SAM Auto Generation
    if generator is not None:
        sam_masks = sam_auto_segment(image, generator, cfg)
        print(f"  🤖 SAM Auto masks: {len(sam_masks)}")
    else:
        # Fallback: watershed + predictor
        separated, ws_n, _ = watershed_separation(no_strokes, cfg)
        print(f"  🌊 Watershed regions: {ws_n}")
        nl,labels,_,_ = cv2.connectedComponentsWithStats(separated)
        sam_masks = []  # predictor path removed for simplicity

    all_m   = size_filter(sam_masks + stroke_masks, image.shape, cfg)
    final   = apply_nms(all_m, cfg, expected_count=expected_glyphs)
    ordered = reading_order(image, final, cfg)
    crops   = save_crops(image, ordered, cfg, image_name=image_path)
    print(f"  ✅ Final glyphs: {len(ordered)}")
    if show:
        visualize_segmentation(image, binary, ordered, crops,
                                title=os.path.basename(image_path))
    return ordered, crops

print("✅  Step 2 complete — Segmentation pipeline ready!")


### Cell 2.10 — 🧪 Test on a Single Random Image

In [ ]:
STELAE_DIR = "/kaggle/input/datasets/ahmedelkelany/egyptian-hieroglyphic-signs-segmentation"

_imgs = sorted(
    _glob.glob(os.path.join(STELAE_DIR, "**", "*.jpg"),  recursive=True) +
    _glob.glob(os.path.join(STELAE_DIR, "**", "*.jpeg"), recursive=True) +
    _glob.glob(os.path.join(STELAE_DIR, "**", "*.png"),  recursive=True) +
    _glob.glob(os.path.join(STELAE_DIR, "**", "*.bmp"),  recursive=True)
)

if not _imgs:
    raise FileNotFoundError(
        f"❌ مفيش صور في {STELAE_DIR}\n"
        "اتأكد إن الـ dataset اتضاف صح في Kaggle → Add Input:\n"
        "  'ahmedelkelany/egyptian-hieroglyphic-signs-segmentation'"
    )

print(f"📂 Found {len(_imgs)} stela images")

test_image = random.choice(_imgs)
# test_image = _imgs[7]  # ← أو حدد رقم معين

print(f"🖼️  Testing: {os.path.basename(test_image)}")
masks, crops = run_single(
    test_image,
    cfg       = seg_cfg,
    generator = sam_generator,
    show      = True
)
print(f"\n✅  Done — {len(masks)} glyphs | {len(crops)} crops saved.")


### Cell 2.11 — 📋 Full Dataset Evaluation (كل الصور) ✅

In [ ]:
def evaluate_dataset(
    dataset_path : str,
    cfg          : SegConfig = None,
    generator    : SamAutomaticMaskGenerator = None,
    max_images   : int  = None,
    save_report  : bool = True,
    report_path  : str  = "/kaggle/working/eval_report.png"
) -> list:
    """
    ✅ Loop كامل على كل الصور في الداتاسيت:
    - يعرض Original + Binary + Glyphs لكل صورة فوراً
    - في الآخر: Bar chart + Histogram + JSON report شامل
    """
    if cfg is None:       cfg       = SegConfig()
    if generator is None: generator = load_sam_auto(cfg)

    _all = []
    for ext in ["*.jpg","*.jpeg","*.png","*.bmp","*.tif","*.tiff"]:
        _all.extend(_glob.glob(os.path.join(dataset_path,"**",ext), recursive=True))
        _all.extend(_glob.glob(os.path.join(dataset_path,ext)))
    all_files = sorted(set(_all))
    if max_images: all_files = all_files[:max_images]

    print(f"{'═'*60}")
    print(f"  📂  Dataset      : {dataset_path}")
    print(f"  🖼️   Total images : {len(all_files)}")
    print(f"{'═'*60}\n")

    results    = []
    all_colors = plt.cm.tab20.colors

    for img_idx, fpath in enumerate(all_files):
        fname = os.path.basename(fpath)
        try:
            image                    = load_image(fpath)
            image                    = remove_border(image, cfg)
            binary, _                = preprocess(image, cfg)
            binary                   = remove_noise(binary, cfg)
            stroke_masks, _          = detect_strokes(binary, cfg)
            sam_masks                = sam_auto_segment(image, generator, cfg)
            all_m                    = size_filter(sam_masks + stroke_masks, image.shape, cfg)
            final                    = apply_nms(all_m, cfg)
            ordered                  = reading_order(image, final, cfg)
            save_crops(image, ordered, cfg, image_name=fpath)
            n_glyphs                 = len(ordered)

            numbered = image.copy()
            for idx, m in enumerate(ordered):
                ys, xs = np.where(m == 1)
                if not len(xs): continue
                x1,y1,x2,y2 = xs.min(),ys.min(),xs.max(),ys.max()
                c = tuple(int(v*255) for v in all_colors[idx%len(all_colors)][:3])
                cv2.rectangle(numbered,(x1,y1),(x2,y2),c,2)
                cv2.putText(numbered,str(idx+1),(x1+2,y1+14),
                            cv2.FONT_HERSHEY_SIMPLEX,0.45,(255,255,255),1)
            status = "✅"

        except Exception as e:
            print(f"  ⚠️  Error on {fname}: {e}")
            n_glyphs = 0; status = f"❌ {str(e)[:50]}"
            image    = np.ones((100,100,3),dtype=np.uint8)*200
            binary   = np.zeros((100,100),dtype=np.uint8)
            numbered = image.copy()

        results.append({"idx":img_idx+1,"filename":fname,"path":fpath,
                         "glyphs":n_glyphs,"status":status})

        # عرض فوري بعد كل صورة
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        axes[0].imshow(image);              axes[0].set_title("① Original",fontsize=11);              axes[0].axis("off")
        axes[1].imshow(binary,cmap="gray"); axes[1].set_title("② Binary (Multi-threshold)",fontsize=11); axes[1].axis("off")
        axes[2].imshow(numbered);           axes[2].set_title(f"③ {n_glyphs} Glyphs ✅",fontsize=11);  axes[2].axis("off")
        short = fname if len(fname)<=45 else "..."+fname[-42:]
        plt.suptitle(
            f"[{img_idx+1}/{len(all_files)}]  {short}   →   {n_glyphs} glyphs   {status}",
            fontsize=12, fontweight="bold"
        )
        plt.tight_layout(); plt.show()
        print(f"  [{img_idx+1:>3}/{len(all_files)}]  {n_glyphs:>3} glyphs   {fname}")

    # ── Summary ──────────────────────────────────────────────────────────────
    ok   = [r for r in results if r["glyphs"] > 0]
    cnts = [r["glyphs"] for r in ok]
    tot  = sum(cnts); avg = tot / max(len(ok), 1)
    errs = len([r for r in results if "❌" in r["status"]])

    print(f"\n{'═'*60}")
    print(f"  📊  DATASET SUMMARY")
    print(f"{'═'*60}")
    print(f"  Total images   : {len(all_files)}")
    print(f"  Successful     : {len(ok)}    Errors: {errs}")
    print(f"  Total glyphs   : {tot}")
    print(f"  Avg / image    : {avg:.1f}    Min: {min(cnts) if cnts else 0}    Max: {max(cnts) if cnts else 0}")
    print(f"{'═'*60}\n")

    # Top 5
    if ok:
        print("  🏆 Top 5 by glyph count:")
        for r in sorted(ok, key=lambda x:-x["glyphs"])[:5]:
            print(f"     {r['filename']:40s} → {r['glyphs']:3d} glyphs")

    # ── Charts ───────────────────────────────────────────────────────────────
    if len(results) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(max(14, len(results)*0.6), 5))
        idxs   = [r["idx"]    for r in results]
        counts = [r["glyphs"] for r in results]
        bar_c  = ["#e74c3c" if c==0 else "#2ecc71" if c>=avg else "#3498db" for c in counts]

        axes[0].bar(idxs, counts, color=bar_c, edgecolor="white", linewidth=0.5)
        axes[0].axhline(avg, color="orange", lw=2, ls="--", label=f"Avg={avg:.1f}")
        axes[0].set_xlabel("Image Index"); axes[0].set_ylabel("# Glyphs")
        axes[0].set_title("Glyph Count per Image", fontweight="bold")
        axes[0].legend(); axes[0].grid(axis="y", alpha=0.3)

        axes[1].hist(counts, bins=max(5,len(set(counts))), color="#9b59b6", edgecolor="white", lw=0.7)
        axes[1].axvline(avg, color="orange", lw=2, ls="--", label=f"Avg={avg:.1f}")
        axes[1].set_xlabel("# Glyphs"); axes[1].set_ylabel("# Images")
        axes[1].set_title("Distribution of Glyph Counts", fontweight="bold")
        axes[1].legend(); axes[1].grid(axis="y", alpha=0.3)

        plt.suptitle(
            f"Dataset Evaluation — {len(all_files)} images | {tot} total glyphs | avg {avg:.1f}/image",
            fontsize=13, fontweight="bold"
        )
        plt.tight_layout()
        if save_report:
            plt.savefig(report_path, dpi=150, bbox_inches="tight")
            print(f"  💾 Report saved → {report_path}")
        plt.show()

    # ── Save JSON report ──────────────────────────────────────────────────────
    json_report = {
        "dataset": dataset_path, "total_images": len(all_files),
        "successful": len(ok), "errors": errs,
        "total_glyphs": tot, "avg_per_image": round(avg, 2),
        "results": results
    }
    json_path = os.path.join("/kaggle/working", "seg_report.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(json_report, f, ensure_ascii=False, indent=2)
    print(f"  💾 JSON report → {json_path}")

    return results


# ══════════════════════════════════════════════════════════════
# ▶️  RUN — كل الصور في الداتاسيت
# ══════════════════════════════════════════════════════════════
STELAE_DIR = "/kaggle/input/datasets/ahmedelkelany/egyptian-hieroglyphic-signs-segmentation"

_check = _glob.glob(os.path.join(STELAE_DIR,"**","*.jpg"), recursive=True) + \
         _glob.glob(os.path.join(STELAE_DIR,"**","*.png"), recursive=True)
if not _check:
    raise FileNotFoundError(
        f"❌ مفيش صور في {STELAE_DIR}\n"
        "اتأكد إن الداتاسيت اتضاف في Kaggle → Add Input:\n"
        "  'ahmedelkelany/egyptian-hieroglyphic-signs-segmentation'"
    )

eval_results = evaluate_dataset(
    dataset_path = STELAE_DIR,
    cfg          = seg_cfg,
    generator    = sam_generator,
    max_images   = None,      # ✅ None = كل الصور | حط رقم لو عايز تختبر أول
    save_report  = True,
    report_path  = "/kaggle/working/eval_report.png"
)
